In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = '/content/drive/MyDrive/Colab Notebooks'

In [ ]:

# 1. Thiết lập cấu hình cơ bản
np.random.seed(42) # Giữ cố định random seed để kết quả luôn giống nhau mỗi lần chạy
n_samples = 5000 # Số lượng ca mổ giả lập

print(f"Đang tiến hành tạo {n_samples} ca mổ giả lập...")

# 2. Sinh dữ liệu Tập Đặc trưng (Features - X)
# Phân bố Tuổi (Age): Trung bình 55, độ lệch chuẩn 15, giới hạn từ 18 đến 90
age = np.clip(np.random.normal(loc=55, scale=15, size=n_samples), 18, 90).astype(int)

# Phân bố BMI: Trung bình 23.5, độ lệch chuẩn 3.5
bmi = np.round(np.clip(np.random.normal(loc=23.5, scale=3.5, size=n_samples), 15, 40), 1)

# Điểm ASA (Từ I đến V): Tỷ lệ thực tế thường tập trung ở I và II
asa_choices = ['I', 'II', 'III', 'IV', 'V']
asa_probs = [0.35, 0.45, 0.15, 0.04, 0.01]
asa_score = np.random.choice(asa_choices, size=n_samples, p=asa_probs)

# Bệnh nền (1: Có, 0: Không)
diabetes = np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]) # 20% tiểu đường
htn = np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3])      # 30% cao huyết áp

# Tính chất mổ: Chương trình (Elective) vs Cấp cứu (Emergency)
surgery_type = np.random.choice(['Chương trình', 'Cấp cứu'], size=n_samples, p=[0.85, 0.15])

# Cận lâm sàng: Bạch cầu (WBC) và Albumin máu
wbc = np.round(np.clip(np.random.normal(loc=8.0, scale=3.5, size=n_samples), 3.0, 25.0), 1)
albumin = np.round(np.clip(np.random.normal(loc=3.8, scale=0.6, size=n_samples), 1.5, 5.5), 1)

# 3. Tạo Biến Mục tiêu (Target - Y) có tính Logic Y khoa
# Chúng ta tạo ra một "Điểm rủi ro ẩn" (Hidden Risk Score) dựa trên các trọng số y khoa giả lập
# ASA cao, Cấp cứu, Tuổi cao, Albumin thấp sẽ làm tăng rủi ro
asa_numeric = np.array([{'I':1, 'II':2, 'III':3, 'IV':4, 'V':5}[x] for x in asa_score])
is_emergency = np.where(surgery_type == 'Cấp cứu', 1, 0)

hidden_risk_score = (
    (age * 0.05) +
    (bmi * 0.1) +
    (asa_numeric * 2.5) +
    (is_emergency * 3.0) +
    (diabetes * 1.5) +
    (htn * 1.0) -
    (albumin * 2.0) + # Albumin càng cao càng tốt nên trừ đi
    (wbc * 0.1) +
    np.random.normal(0, 2, n_samples) # Thêm nhiễu ngẫu nhiên (Noise) để AI không học thuộc lòng
)

# Chuyển đổi điểm rủi ro ẩn thành nhãn 0/1 (Nguy cơ thấp / Nguy cơ cao)
# Cố tình ép tỷ lệ Imbalanced: Chỉ lấy Top 10% điểm rủi ro cao nhất làm nhóm Biến chứng (Nhãn 1)
threshold = np.percentile(hidden_risk_score, 90) # Lấy mốc phân vị 90%
high_risk_flag = np.where(hidden_risk_score >= threshold, 1, 0)

# 4. Gom dữ liệu vào Pandas DataFrame
df = pd.DataFrame({
    'Visit_ID': [f"SURG_{str(i).zfill(5)}" for i in range(1, n_samples + 1)],
    'Age': age,
    'BMI': bmi,
    'ASA_Score': asa_score,
    'Has_Diabetes': diabetes,
    'Has_HTN': htn,
    'Surgery_Type': surgery_type,
    'PreOp_WBC': wbc,
    'PreOp_Albumin': albumin,
    'High_Risk_Flag': high_risk_flag
})

# 5. Cố tình tạo Dữ liệu Khuyết thiếu (Missing Values) để thực hành tiền xử lý
# Cho 10% WBC bị thiếu và 15% Albumin bị thiếu (vì xét nghiệm Albumin ít phổ biến hơn WBC)
missing_wbc_idx = np.random.choice(df.index, size=int(n_samples * 0.10), replace=False)
missing_alb_idx = np.random.choice(df.index, size=int(n_samples * 0.15), replace=False)

df.loc[missing_wbc_idx, 'PreOp_WBC'] = np.nan
df.loc[missing_alb_idx, 'PreOp_Albumin'] = np.nan

# 6. Lưu ra file CSV và in tóm tắt
df.to_csv('/content/drive/MyDrive/Colab Notebooks/mock_surgery_data.csv', index=False)

print("\n--- HOÀN THÀNH TẠO DỮ LIỆU ---")
print(f"File 'mock_surgery_data.csv' đã được lưu vào hệ thống Colab.")
print(f"Tổng số ca mổ: {len(df)}")
print(f"Số ca có biến chứng (Nhãn 1): {df['High_Risk_Flag'].sum()} ca ({df['High_Risk_Flag'].mean()*100:.1f}%)")
print("\nXem trước 5 dòng đầu tiên:")
display(df.head())

Đang tiến hành tạo 5000 ca mổ giả lập...

--- HOÀN THÀNH TẠO DỮ LIỆU ---
File 'mock_surgery_data.csv' đã được lưu vào hệ thống Colab.
Tổng số ca mổ: 5000
Số ca có biến chứng (Nhãn 1): 500 ca (10.0%)

Xem trước 5 dòng đầu tiên:


,Visit_ID,Age,BMI,ASA_Score,Has_Diabetes,Has_HTN,Surgery_Type,PreOp_WBC,PreOp_Albumin,High_Risk_Flag
0,SURG_00001,62,22.0,I,0,0,Chương trình,10.1,3.0,0
1,SURG_00002,52,21.9,I,0,0,Chương trình,3.7,4.4,0
2,SURG_00003,64,17.2,I,0,1,Chương trình,8.4,3.7,0
3,SURG_00004,77,22.3,I,0,1,Chương trình,4.7,2.9,0
4,SURG_00005,51,26.1,II,0,0,Chương trình,3.1,NaN,0
